# 3.1 Object caption evaluation

In [ ]:
import json

gt_json_path = "0a7cc12c0e.json"
pred_json_path = "object_captions_1.json"

with open(gt_json_path, "r") as f:
    gt_data = json.load(f)

with open(pred_json_path, "r") as f:
    pred_data = json.load(f)

gt_labels = {
    int(obj_id): obj_info["label"].strip().lower()
    for obj_id, obj_info in gt_data["objects"].items()
    if "label" in obj_info
}

pred_labels = {
    int(obj["object_id"]): obj["category"].strip().lower()
    for obj in pred_data
}

matches = []
for gt_id, gt_label in gt_labels.items():
    for pred_id, pred_label in pred_labels.items():
        if gt_label == pred_label:
            matches.append({
                "gt_id": gt_id,
                "gt_label": gt_label,
                "pred_id": pred_id,
                "pred_label": pred_label
            })

import pandas as pd
df_matches = pd.DataFrame(matches)
print("匹配到的 object：")
display(df_matches)

matched_gt_ids = set([m['gt_id'] for m in matches])
print(f"共匹配到 {len(matched_gt_ids)} 个 ground truth object。")



匹配到的 object：


,gt_id,gt_label,pred_id,pred_label
0,34,shelf,2,shelf
1,34,shelf,34,shelf
2,34,shelf,41,shelf
3,34,shelf,48,shelf
4,34,shelf,61,shelf
...,...,...,...,...
202,16,sink,252,sink
203,16,sink,286,sink
204,16,sink,293,sink
205,16,sink,294,sink


共匹配到 19 个 ground truth object。


In [2]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     /home/vlm_caption/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/vlm_caption/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [19]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.tokenize import word_tokenize
from collections import defaultdict

smoothie = SmoothingFunction().method4

# 按 GT id 分组匹配项
gt_to_preds = defaultdict(list)
for m in matches:
    gt_to_preds[m['gt_id']].append(m['pred_id'])

bleu_scores = []

# 遍历每个唯一 GT object
for gt_id, pred_ids in gt_to_preds.items():
    gt_captions = gt_data["objects"][str(gt_id)]["summarized_local_caption_16"]
    references = [word_tokenize(ref.lower()) for ref in gt_captions]

    best_bleu = -1
    for pred_id in pred_ids:
        pred_obj = next(obj for obj in pred_data if obj["object_id"] == pred_id)
        candidate = word_tokenize(pred_obj["object_caption"].lower())

        # 这个 pred_id 的 BLEU 取和所有 refs 中的最大
        pred_bleu = max([
            sentence_bleu([word_tokenize(ref.lower())], candidate, smoothing_function=smoothie)
            for ref in gt_captions
        ])
        best_bleu = max(best_bleu, pred_bleu)  # 在多个 pred_id 中选最大

    bleu_scores.append(best_bleu)

# 打印每个 GT 的 BLEU
for i, score in enumerate(bleu_scores):
    print(f"GT #{i+1}: BLEU = {score:.4f}")

# 打印平均值
avg_bleu = sum(bleu_scores) / len(bleu_scores)
print(f"\n平均 BLEU 分数: {avg_bleu:.4f}")


GT #1: BLEU = 0.1773
GT #2: BLEU = 0.0342
GT #3: BLEU = 0.0726
GT #4: BLEU = 0.0951
GT #5: BLEU = 0.2254
GT #6: BLEU = 0.0388
GT #7: BLEU = 0.0187
GT #8: BLEU = 0.1090
GT #9: BLEU = 0.1188
GT #10: BLEU = 0.0506
GT #11: BLEU = 0.0801
GT #12: BLEU = 0.1326
GT #13: BLEU = 0.0916
GT #14: BLEU = 0.0243
GT #15: BLEU = 0.0705
GT #16: BLEU = 0.1900
GT #17: BLEU = 0.0972
GT #18: BLEU = 0.3204
GT #19: BLEU = 0.1985

平均 BLEU 分数: 0.1129


In [ ]:
import evaluate
rouge = evaluate.load("rouge")

rouge_l_scores = []

# 遍历每个唯一的 GT object
for gt_id, pred_ids in gt_to_preds.items():
    gt_captions = gt_data["objects"][str(gt_id)]["summarized_local_caption_16"]

    best_rouge = -1
    for pred_id in pred_ids:
        pred_obj = next(obj for obj in pred_data if obj["object_id"] == pred_id)
        pred_caption = pred_obj["object_caption"]

        # 计算这个 prediction caption 与每个 reference 的 ROUGE-L，取最大
        scores = [
            rouge.compute(predictions=[pred_caption], references=[ref])["rougeL"]
            for ref in gt_captions
        ]
        rouge_score = max(scores)
        best_rouge = max(best_rouge, rouge_score)

    rouge_l_scores.append(best_rouge)

# 打印每个 GT 的 ROUGE-L 分数
for i, score in enumerate(rouge_l_scores):
    print(f"GT #{i+1}: ROUGE-L = {score:.4f}")

# 平均
avg_rouge = sum(rouge_l_scores) / len(rouge_l_scores)
print(f"\n平均 ROUGE-L 分数: {avg_rouge:.4f}")


GT #1: ROUGE-L = 0.4516
GT #2: ROUGE-L = 0.2000
GT #3: ROUGE-L = 0.3902
GT #4: ROUGE-L = 0.4118
GT #5: ROUGE-L = 0.4865
GT #6: ROUGE-L = 0.1538
GT #7: ROUGE-L = 0.0800
GT #8: ROUGE-L = 0.4167
GT #9: ROUGE-L = 0.2667
GT #10: ROUGE-L = 0.2424
GT #11: ROUGE-L = 0.3243
GT #12: ROUGE-L = 0.4375
GT #13: ROUGE-L = 0.3784
GT #14: ROUGE-L = 0.1463
GT #15: ROUGE-L = 0.2581
GT #16: ROUGE-L = 0.4571
GT #17: ROUGE-L = 0.4000
GT #18: ROUGE-L = 0.5556
GT #19: ROUGE-L = 0.3784

平均 ROUGE-L 分数: 0.3387


In [5]:
import evaluate
meteor = evaluate.load("meteor")

meteor_scores = []

# 遍历每个唯一的 GT object
for gt_id, pred_ids in gt_to_preds.items():
    gt_captions = gt_data["objects"][str(gt_id)]["summarized_local_caption_16"]

    best_meteor = -1
    for pred_id in pred_ids:
        pred_obj = next(obj for obj in pred_data if obj["object_id"] == pred_id)
        pred_caption = pred_obj["object_caption"]

        # 分别与 5 个 reference 比较，取最大 METEOR
        scores = [
            meteor.compute(predictions=[pred_caption], references=[ref])["meteor"]
            for ref in gt_captions
        ]
        meteor_score = max(scores)
        best_meteor = max(best_meteor, meteor_score)

    meteor_scores.append(best_meteor)

# 打印每个 GT 的 METEOR
for i, score in enumerate(meteor_scores):
    print(f"GT #{i+1}: METEOR = {score:.4f}")

# 平均
avg_meteor = sum(meteor_scores) / len(meteor_scores)
print(f"\n平均 METEOR 分数: {avg_meteor:.4f}")


[nltk_data] Downloading package wordnet to
[nltk_data]     /home/vlm_caption/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/vlm_caption/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /home/vlm_caption/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


GT #1: METEOR = 0.5775
GT #2: METEOR = 0.1310
GT #3: METEOR = 0.4567
GT #4: METEOR = 0.5989
GT #5: METEOR = 0.6402
GT #6: METEOR = 0.1587
GT #7: METEOR = 0.1389
GT #8: METEOR = 0.5437
GT #9: METEOR = 0.2881
GT #10: METEOR = 0.3040
GT #11: METEOR = 0.3670
GT #12: METEOR = 0.5287
GT #13: METEOR = 0.4644
GT #14: METEOR = 0.3367
GT #15: METEOR = 0.3055
GT #16: METEOR = 0.5989
GT #17: METEOR = 0.4852
GT #18: METEOR = 0.7315
GT #19: METEOR = 0.4329

平均 METEOR 分数: 0.4257


In [7]:
from pycocoevalcap.cider.cider import Cider

cider = Cider()

cider_scores = []

for gt_id, pred_ids in gt_to_preds.items():
    gt_captions = gt_data["objects"][str(gt_id)]["summarized_local_caption_16"]

    best_cider = -1
    for pred_id in pred_ids:
        pred_obj = next(obj for obj in pred_data if obj["object_id"] == pred_id)
        pred_caption = pred_obj["object_caption"]

        # 构造 CIDEr 输入格式
        refs_dict = {0: gt_captions}         # list of 5 references
        preds_dict = {0: [pred_caption]}     # list of 1 prediction

        score, _ = cider.compute_score(refs_dict, preds_dict)
        best_cider = max(best_cider, score)

    cider_scores.append(best_cider)

# 输出每个 GT 的 CIDEr 分数
for i, score in enumerate(cider_scores):
    print(f"GT #{i+1}: CIDEr = {score:.4f}")

# 平均
avg_cider = sum(cider_scores) / len(cider_scores)
print(f"\n平均 CIDEr 分数: {avg_cider:.4f}")


GT #1: CIDEr = 0.0000
GT #2: CIDEr = 0.0000
GT #3: CIDEr = 0.0000
GT #4: CIDEr = 0.0000
GT #5: CIDEr = 0.0000
GT #6: CIDEr = 0.0000
GT #7: CIDEr = 0.0000
GT #8: CIDEr = 0.0000
GT #9: CIDEr = 0.0000
GT #10: CIDEr = 0.0000
GT #11: CIDEr = 0.0000
GT #12: CIDEr = 0.0000
GT #13: CIDEr = 0.0000
GT #14: CIDEr = 0.0000
GT #15: CIDEr = 0.0000
GT #16: CIDEr = 0.0000
GT #17: CIDEr = 0.0000
GT #18: CIDEr = 0.0000
GT #19: CIDEr = 0.0000

平均 CIDEr 分数: 0.0000


In [9]:
# 手动测试是否 scorer 正常
test_refs = {0: [
    "a drying rack with a metal frame",
    "white and blue drying rack with wire mesh",
    "drying rack structure with curved rods",
    "blue frame drying rack with white surface",
    "rectangular metal rack for drying"
]}
test_preds = {0: ["a white and blue metal drying rack with mesh structure"]}

score, _ = cider.compute_score(test_refs, test_preds)
print(f"[测试] CIDEr 分数: {score:.4f}")


[测试] CIDEr 分数: 0.0000


In [10]:
import sacrebleu

sacrebleu_scores = []

for gt_id, pred_ids in gt_to_preds.items():
    gt_captions = gt_data["objects"][str(gt_id)]["summarized_local_caption_16"]

    best_bleu = -1
    for pred_id in pred_ids:
        pred_obj = next(obj for obj in pred_data if obj["object_id"] == pred_id)
        pred_caption = pred_obj["object_caption"]

        # sacrebleu 需要 refs 是 list of list（多组 reference）
        score = sacrebleu.corpus_bleu(
            [pred_caption],  # list of hypothesis
            [gt_captions]    # list of list of references
        ).score  # 返回的是 0 ~ 100 的分数
        best_bleu = max(best_bleu, score)

    sacrebleu_scores.append(best_bleu)

# 打印每个 GT 的 BLEU
for i, score in enumerate(sacrebleu_scores):
    print(f"GT #{i+1}: sacreBLEU = {score:.2f}")

# 平均
avg_sacrebleu = sum(sacrebleu_scores) / len(sacrebleu_scores)
print(f"\n平均 sacreBLEU 分数: {avg_sacrebleu:.2f}")


GT #1: sacreBLEU = 11.33
GT #2: sacreBLEU = 2.27
GT #3: sacreBLEU = 4.30
GT #4: sacreBLEU = 3.65
GT #5: sacreBLEU = 8.79
GT #6: sacreBLEU = 2.39
GT #7: sacreBLEU = 2.53
GT #8: sacreBLEU = 8.79
GT #9: sacreBLEU = 7.95
GT #10: sacreBLEU = 2.05
GT #11: sacreBLEU = 5.41
GT #12: sacreBLEU = 13.26
GT #13: sacreBLEU = 4.81
GT #14: sacreBLEU = 1.73
GT #15: sacreBLEU = 4.72
GT #16: sacreBLEU = 9.67
GT #17: sacreBLEU = 11.20
GT #18: sacreBLEU = 9.88
GT #19: sacreBLEU = 4.71

平均 sacreBLEU 分数: 6.29


In [11]:
from bert_score import score as bertscore

bert_scores = []

for gt_id, pred_ids in gt_to_preds.items():
    gt_captions = gt_data["objects"][str(gt_id)]["summarized_local_caption_16"]

    best_score = -1
    for pred_id in pred_ids:
        pred_obj = next(obj for obj in pred_data if obj["object_id"] == pred_id)
        pred_caption = pred_obj["object_caption"]

        # 与每个 reference 分别计算 BERTScore
        preds = [pred_caption] * len(gt_captions)
        refs = gt_captions

        P, R, F1 = bertscore(preds, refs, lang="en", verbose=False)
        best_f1 = max(F1).item()  # 取最大 F1 作为该 prediction 的代表

        best_score = max(best_score, best_f1)  # 所有 prediction 中取最大

    bert_scores.append(best_score)

# 打印每个 GT 的 BERTScore
for i, score_val in enumerate(bert_scores):
    print(f"GT #{i+1}: BERTScore (F1) = {score_val:.4f}")

# 平均
avg_bert = sum(bert_scores) / len(bert_scores)
print(f"\n平均 BERTScore (F1): {avg_bert:.4f}")


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You sho

GT #1: BERTScore (F1) = 0.9144
GT #2: BERTScore (F1) = 0.8663
GT #3: BERTScore (F1) = 0.9091
GT #4: BERTScore (F1) = 0.9085
GT #5: BERTScore (F1) = 0.9315
GT #6: BERTScore (F1) = 0.8716
GT #7: BERTScore (F1) = 0.8652
GT #8: BERTScore (F1) = 0.9044
GT #9: BERTScore (F1) = 0.8711
GT #10: BERTScore (F1) = 0.8830
GT #11: BERTScore (F1) = 0.9054
GT #12: BERTScore (F1) = 0.9065
GT #13: BERTScore (F1) = 0.9116
GT #14: BERTScore (F1) = 0.8588
GT #15: BERTScore (F1) = 0.8839
GT #16: BERTScore (F1) = 0.9122
GT #17: BERTScore (F1) = 0.9172
GT #18: BERTScore (F1) = 0.9320
GT #19: BERTScore (F1) = 0.9023

平均 BERTScore (F1): 0.8976


In [16]:
from collections import defaultdict

# 临时聚合：gt_id -> 多个分数
gt_to_metrics = defaultdict(list)

# 遍历所有 match 和对应分数
for m, bleu, rouge, meteor, bert in zip(matches, bleu_scores, rouge_l_scores, meteor_scores, bert_scores):
    gt_id = m['gt_id']
    label = m['gt_label']
    gt_to_metrics[gt_id].append({
        "label": label,
        "bleu": bleu,
        "rouge": rouge,
        "meteor": meteor,
        "bert": bert
    })

# 对每个 gt_id 取最大分
first_5_entries = []
for gt_id, metric_list in list(gt_to_metrics.items())[:5]:
    best = {
        "label": metric_list[0]["label"],
        "bleu": max(m["bleu"] for m in metric_list),
        "rouge": max(m["rouge"] for m in metric_list),
        "meteor": max(m["meteor"] for m in metric_list),
        "bert": max(m["bert"] for m in metric_list),
    }
    first_5_entries.append(best)

# 计算平均值
avg_entry = {
    "label": "Average",
    "bleu": sum(bleu_scores) / len(bleu_scores),
    "rouge": sum(rouge_l_scores) / len(rouge_l_scores),
    "meteor": sum(meteor_scores) / len(meteor_scores),
    "bert": sum(bert_scores) / len(bert_scores)
}

# 输出格式化表格
print(f"{'Label':<20} {'BLEU':>6} {'ROUGE-L':>8} {'METEOR':>8} {'BERTScore':>10}")
print("-" * 56)
for entry in first_5_entries:
    print(f"{entry['label']:<20} {entry['bleu']:.4f}   {entry['rouge']:.4f}   {entry['meteor']:.4f}   {entry['bert']:.4f}")
print()
print(f"{avg_entry['label']:<20} {avg_entry['bleu']:.4f}   {avg_entry['rouge']:.4f}   {avg_entry['meteor']:.4f}   {avg_entry['bert']:.4f}")


Label                  BLEU  ROUGE-L   METEOR  BERTScore
--------------------------------------------------------
shelf                0.3204   0.5556   0.7315   0.9320

Average              0.1129   0.3387   0.4257   0.8976


In [20]:
# 提取所有唯一 label 及其对应唯一 gt_id（去重）
unique_label_ids = {}
for match in matches:
    label = match["gt_label"]
    gt_id = match["gt_id"]
    if label not in unique_label_ids:
        unique_label_ids[label] = gt_id

# 打印结果
for label, gt_id in unique_label_ids.items():
    print(f"{label:<15} → {gt_id}")


shelf           → 34
table           → 7
rug             → 37
door            → 61
chair           → 68
kitchen cabinet → 70
laptop          → 31
pillow          → 26
shoes           → 39
pot             → 5
bottle          → 6
toilet          → 12
sink            → 16


In [21]:
# 假设 bleu_scores 和 gt_to_preds 已经存在

# 按顺序取出每个唯一 gt_id
gt_ids = list(gt_to_preds.keys())

# 打印每个 BLEU 分数
for i, (gt_id, score) in enumerate(zip(gt_ids, bleu_scores)):
    print(f"GT ID {gt_id:<5} | BLEU = {score:.4f}")


GT ID 34    | BLEU = 0.1773
GT ID 7     | BLEU = 0.0342
GT ID 37    | BLEU = 0.0726
GT ID 61    | BLEU = 0.0951
GT ID 68    | BLEU = 0.2254
GT ID 70    | BLEU = 0.0388
GT ID 71    | BLEU = 0.0187
GT ID 73    | BLEU = 0.1090
GT ID 31    | BLEU = 0.1188
GT ID 67    | BLEU = 0.0506
GT ID 26    | BLEU = 0.0801
GT ID 27    | BLEU = 0.1326
GT ID 28    | BLEU = 0.0916
GT ID 39    | BLEU = 0.0243
GT ID 5     | BLEU = 0.0705
GT ID 6     | BLEU = 0.1900
GT ID 13    | BLEU = 0.0972
GT ID 12    | BLEU = 0.3204
GT ID 16    | BLEU = 0.1985
